# Pipeline documental

Gera chunks rastreáveis e sanitizados para indexação. Os TXT originais não são alterados.

In [14]:
from pathlib import Path
import re
import sys

from transformers import AutoTokenizer

def find_repository_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "documents").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do repositório.")

REPOSITORY_ROOT = find_repository_root()
sys.path.insert(0, str(REPOSITORY_ROOT))

from src.ingestion.chunking import CPF_PATTERN, REMUNERATION_PATTERN, build_chunks
from src.ingestion.embeddings import generate_embeddings


In [15]:
MODEL_NAME = "BAAI/bge-m3"
MAX_TOKENS = 350
OVERLAP_TOKENS = 50
DATASET_DIR = (
    REPOSITORY_ROOT
    / "documents/downloads/novatech_proximos_passos_lpg"
    / "dataset_saneado_corrigido/dataset_saneado"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
all_chunks = build_chunks(
    index_path=DATASET_DIR / "documentos_index.csv",
    data_dir=DATASET_DIR,
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
    overlap_tokens=OVERLAP_TOKENS,
)

print(f"Chunks gerados: {len(all_chunks)}")
print(f"Documentos processados: {len({chunk['doc_id'] for chunk in all_chunks})}")


Chunks gerados: 113
Documentos processados: 60


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_DIMENSION = 1024
EMBEDDING_BATCH_SIZE = 32
embedding_model = SentenceTransformer(MODEL_NAME)
chunks_with_embeddings = generate_embeddings(
    all_chunks,
    embedding_model,
    embedding_dimension=EMBEDDING_DIMENSION,
    batch_size=EMBEDDING_BATCH_SIZE,
)

print(f"Embeddings gerados: {len(chunks_with_embeddings)}")
print(f"Dimensão: {len(chunks_with_embeddings[0]['embedding'])}")


In [16]:
assert all(chunk["token_count"] <= MAX_TOKENS for chunk in all_chunks)
assert all(chunk["texto"].strip() for chunk in all_chunks)
assert all(chunk["doc_id"].startswith("DOC-") for chunk in all_chunks)
assert all(not CPF_PATTERN.search(str(chunk["texto"])) for chunk in all_chunks)
assert all(not REMUNERATION_PATTERN.search(str(chunk["texto"])) for chunk in all_chunks)
assert all(len(chunk["embedding"]) == EMBEDDING_DIMENSION for chunk in chunks_with_embeddings)

print("Validações de tamanho, conteúdo e IDs canônicos concluídas.")


Validações de tamanho, conteúdo e IDs canônicos concluídas.
